# Can we predict a warranty claim within a year?
Protocol: `protocol.yaml`. Fixed holdout (`holdout.csv`, 0-based row positions in data.csv), 5-fold stratified CV on training rows, choice by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data.csv")
assert hashlib.sha256(open("data.csv","rb").read()).hexdigest()=="25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4"
hold_idx = pd.read_csv("holdout.csv")["row"].to_numpy()
assert len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
train, hold = df.iloc[train_idx], df.iloc[hold_idx]
target="claim_within_year"
print(len(train), len(hold), train[target].mean(), hold[target].mean())

## Features
`device_serial` is excluded (identifier, per protocol). The other columns (channel, device age at sale, price, reported daily use at registration) are all known at sale time, so none is a post-outcome leak.

In [ ]:
features=["channel","device_age_years","price_eur","daily_hours"]
cat=["channel"]; num=[f for f in features if f not in cat]
def prep(scale=True):
    return ColumnTransformer([("c",OneHotEncoder(handle_unknown="ignore"),cat),
                              ("n",StandardScaler() if scale else "passthrough",num)])
X, y = train[features], train[target]
X_h, y_h = hold[features], hold[target]

## Candidates (named before any scoring)

In [ ]:
candidates = {
 "baseline_prior": make_pipeline(prep(), DummyClassifier(strategy="prior")),
 "logreg_C1": make_pipeline(prep(), LogisticRegression(C=1.0, max_iter=1000)),
 "logreg_C0.1": make_pipeline(prep(), LogisticRegression(C=0.1, max_iter=1000)),
 "random_forest": make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
 "hist_gb": make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
print(list(candidates))

## Cross-validation on training rows (preprocessing is inside the pipeline, so it is fit per fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_folds = [[[int(train_idx[i]) for i in tr], [int(train_idx[i]) for i in va]] for tr, va in cv.split(X, y)]  # [fit rows, validation rows] per fold, data.csv positions
cv_scores = {n: cross_val_score(m, X, y, cv=cv, scoring="roc_auc").tolist() for n,m in candidates.items()}
summary = pd.DataFrame({n:{"mean":np.mean(s),"std":np.std(s)} for n,s in cv_scores.items()}).T
print(summary.sort_values("mean",ascending=False))
chosen = summary["mean"].idxmax(); print("chosen:", chosen)

## Refit chosen candidate on all training rows; score holdout once

In [ ]:
final = candidates[chosen].fit(X, y)
holdout_auc = roc_auc_score(y_h, final.predict_proba(X_h)[:,1])
print("holdout roc_auc:", holdout_auc)

In [ ]:
evidence = {
 "features": features,
 "candidates": list(candidates),
 "cv_folds": cv_folds,
 "cv_scores": cv_scores,
 "chosen": chosen,
 "holdout_rows": [int(i) for i in hold_idx],
 "holdout_scores": {chosen: holdout_auc},
}
json.dump(evidence, open("evidence.json","w"))